# 04 — Features de vendedor (`seller_id`)

Mide cuánto suman las features de vendedor por encima de todo lo confirmado hasta ahora (B+C+D+E+título). 27.949 vendedores en train, mediana de 1 publicación por vendedor, 71,6% del test tiene un vendedor que también aparece en train (consistente con el ~72% de la referencia inicial).

Dos features sin target (`seller_n_listings`, `seller_avg_log_price`) y una con target encoding *leave-one-out* (`seller_pct_new`) — medidas en bloques separados para ver si el target encoding aporta algo por encima de los agregados simples, o si todo el salto viene de ahí.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import warnings

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

from features import aplanar_anidadas, agregar_logs, CapeoPorMoneda, VendedorEncoder

SEED = 42

train = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)
train = aplanar_anidadas(train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

modelo_fijo = HistGradientBoostingClassifier(
    learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED
)

resultados_bloques = []


def evaluar_bloque(nombre, columnas_categoricas, columnas_numericas, usar_tfidf=False,
                    max_features_tfidf=500, vendedor=None):
    """vendedor: None (no agrega nada), 'simple' (n_listings + avg_price,
    sin target) o 'target' (+ seller_pct_new, con leave-one-out). VendedorEncoder
    vive en features.py (promovido desde acá, mismo lugar que CapeoPorMoneda)."""
    transformers = [
        ("onehot", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
        ("num", "passthrough", columnas_numericas),
    ]
    if usar_tfidf:
        transformers.append(("tfidf", TfidfVectorizer(max_features=max_features_tfidf), "title"))
    if vendedor == "simple":
        transformers.append(("vendedor", VendedorEncoder(incluir_target=False), ["seller_id", "log_price"]))
    elif vendedor == "target":
        transformers.append(("vendedor", VendedorEncoder(incluir_target=True), ["seller_id", "log_price"]))

    preprocesador = ColumnTransformer(transformers, sparse_threshold=0)
    pipeline = Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", preprocesador),
        ("modelo", modelo_fijo),
    ])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        resultados = cross_validate(pipeline, train, y, cv=cv, scoring=["accuracy", "roc_auc"])

    fila = {
        "bloque": nombre,
        "accuracy_media": resultados["test_accuracy"].mean(),
        "accuracy_std": resultados["test_accuracy"].std(),
        "roc_auc_media": resultados["test_roc_auc"].mean(),
    }
    resultados_bloques.append(fila)
    print(nombre, "-> accuracy:", round(fila["accuracy_media"], 4), "+/-", round(fila["accuracy_std"], 4),
          "| roc_auc:", round(fila["roc_auc_media"], 4))
    return fila

## Referencia — Base + título (B+C+D+E+H)

Reproduce el mejor resultado confirmado hasta ahora (`03_titulo.ipynb`, bloque H), para comparar contra los bloques de vendedor.

In [ ]:
columnas_categoricas = ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]
columnas_numericas = [
    "log_price", "log_available_quantity", "automatic_relist",
    "sold_quantity", "initial_quantity",
    "n_fotos", "n_atributos", "n_variaciones", "tiene_video", "tiene_garantia",
    "free_shipping", "local_pick_up", "n_metodos_pago", "accepts_mercadopago",
    "es_tienda_oficial", "tiene_descuento", "tiene_oferta", "tiene_parent_item",
]

evaluar_bloque("Base + título", columnas_categoricas, columnas_numericas, usar_tfidf=True)

## Bloque I — Vendedor simple (sin target)

Suma `seller_n_listings` (log1p de la cantidad de publicaciones del vendedor) y `seller_avg_log_price` (precio log promedio del vendedor). Ninguna usa el target — sin riesgo de leakage.

In [ ]:
evaluar_bloque("I - + Vendedor simple", columnas_categoricas, columnas_numericas, usar_tfidf=True, vendedor="simple")

## Bloque J — + Vendedor con target encoding

Suma `seller_pct_new` (leave-one-out) al bloque I.

In [ ]:
evaluar_bloque("J - + Vendedor target encoding", columnas_categoricas, columnas_numericas, usar_tfidf=True, vendedor="target")

## Tabla final y conclusión

In [ ]:
tabla_vendedor = pd.DataFrame(resultados_bloques).set_index("bloque")
tabla_vendedor["ganancia_accuracy"] = tabla_vendedor["accuracy_media"].diff()
tabla_vendedor

**Conclusión.** Las dos features de vendedor aportan, y las dos por separado: el **bloque I** (sin target, `seller_n_listings` + `seller_avg_log_price`) suma **+0,0116** de accuracy — confirma que "cuántas publicaciones tiene y a qué precio vende" ya distingue bastante entre tienda/revendedor (mayormente `new`) y particular (mayormente `used`), sin ningún riesgo de leakage. El **bloque J** (+ `seller_pct_new` con *leave-one-out*) suma otros **+0,0255** encima de eso — el aporte individual más grande que medimos en las 3 etapas (más que el título solo). Ambos muy por encima del desvío entre folds (0,002).

En conjunto, desde "Base + título" (0,8813) hasta acá (0,9184) son **+0,0371** de accuracy — y conviene remarcarlo en la presentación: la ganancia de `seller_pct_new` es grande precisamente porque hay un patrón de negocio fuerte y real (vendedores "tienda" venden sobre todo `new`, particulares sobre todo `used`), no un error de cálculo — pero como la mediana es 1 publicación por vendedor, gran parte de esa señal viene del ~30% de vendedores con varias publicaciones; para los demás, el feature cae al promedio global de respaldo y no aporta nada puntual.

**Resumen de las 3 etapas de features (notebooks 02, 03, 04):**

| Etapa | Accuracy CV | Ganancia |
|---|---|---|
| 5 variables originales | 0,825 | — |
| + B+C+D+E (`02_features.ipynb`) | 0,853 | +0,028 |
| + título (`03_titulo.ipynb`) | 0,881 | +0,028 |
| + vendedor simple (`04_vendedor.ipynb`, bloque I) | 0,893 | +0,012 |
| + vendedor target encoding (`04_vendedor.ipynb`, bloque J) | **0,918** | +0,026 |

**Siguiente paso:** sumar todo esto a `train.ipynb`. El `VendedorEncoder` es código nuevo y reutilizable — como `CapeoPorMoneda`, tendría sentido que viva en `features.py` en vez de quedar solo acá. Y como todo este salto es grande, muy probablemente cambien los mejores hiperparámetros de cada modelo — hay que volver a tunear todo, no solo refittear.